In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from pybaseball import statcast, playerid_lookup

In [2]:
START_DATE = '2023-03-30'
END_DATE = '2026-09-27'
TEST_START_DATE = '2026-03-25'
CACHE_DIR = Path('data')
CACHE_PATH = CACHE_DIR / f'statcast_{START_DATE}_{END_DATE}.parquet'

if CACHE_PATH.exists():
    raw = pd.read_parquet(CACHE_PATH)
    print(f'Loaded {len(raw):,} pitches from {CACHE_PATH}')
else:
    CACHE_DIR.mkdir(parents = True, exist_ok = True)
    raw = statcast(START_DATE, END_DATE)
    raw.to_parquet(CACHE_PATH, index = False)
    print(f'Downloaded and cached {len(raw):,} pitches to {CACHE_PATH}')

Loaded 3,018,395 pitches from data/statcast_2023-03-30_2026-09-27.parquet


In [3]:
bbe = {'field_out', 'single', 'double', 'home_run', 'force_out', 'grounded_into_double_play',
       'field_error', 'sac_fly', 'triple', 'fielders_choice', 'double_play', 'fielders_choice_out',
       'sac_fly_double_play', 'triple_play'}

In [6]:
contact = raw.loc[
    (raw.events.isin(bbe)) & (raw.game_type == 'R'),
    ['game_date', 'batter', 'events', 'launch_speed', 
     'launch_angle', 'stand', 'p_throws', 'pitch_type']]

contact['is_hit'] = contact["events"].isin(
    {"single", "double", "triple", "home_run"}
).astype(int)

contact.head()

,game_date,batter,events,launch_speed,launch_angle,stand,p_throws,pitch_type,is_hit
3,2026-09-27,811965,field_out,90.1,45,R,R,SL,0
4,2026-09-27,695720,field_out,68.5,46,R,R,SL,0
10,2026-09-27,664774,field_out,64.4,-53,L,L,ST,0
13,2026-09-27,665161,field_out,88.4,30,R,L,FF,0
19,2026-09-27,678489,field_out,79.6,82,R,L,FF,0


In [7]:
model_data = contact.dropna(
    subset=["launch_speed", "launch_angle", "stand", "p_throws", "pitch_type", "is_hit"]
).copy()

model_data['launch_speed_x_launch_angle'] = model_data['launch_speed'] * model_data['launch_angle']

print(f"Contact events: {len(contact):,}")
print(f"Complete rows: {len(model_data):,}")
print(f"Observed batting average: {model_data['is_hit'].mean():.3f}")
print("Missing values by field:")
print(contact.isna().sum().sort_values(ascending=False))

Contact events: 495,813
Complete rows: 494,102
Observed batting average: 0.324
Missing values by field:
launch_speed    1659
launch_angle    1358
pitch_type       187
game_date          0
batter             0
events             0
stand              0
p_throws           0
is_hit             0
dtype: int64


In [8]:
model_data.game_date = pd.to_datetime(model_data.game_date)
test_start_date = pd.Timestamp(TEST_START_DATE)
train_data = model_data[model_data['game_date'] < test_start_date].copy()
test_data = model_data[model_data['game_date'] >= test_start_date].copy()

In [9]:
numeric_features = ['launch_speed', 'launch_angle', 'launch_speed_x_launch_angle']
categorical_features = ['stand', 'p_throws', 'pitch_type']
features = numeric_features + categorical_features

X_train = train_data[features]
X_test = test_data[features]
y_train = train_data['is_hit']
y_test = test_data['is_hit']

print(f"Training rows: {len(train_data):,}")
print(f"Test rows: {len(test_data):,}")
print(f"Training dates: {train_data['game_date'].min().date()} to {train_data['game_date'].max().date()}")
print(f"Test dates: {test_data['game_date'].min().date()} to {test_data['game_date'].max().date()}")

Training rows: 370,720
Test rows: 123,382
Training dates: 2023-03-30 to 2025-09-28
Test dates: 2026-03-25 to 2026-09-27


In [10]:
#Baseline model
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy = 'prior')
baseline.fit(X_train, y_train)
baseline_probs = baseline.predict_proba(X_test)[:, 1]

In [11]:
#Simple Logistic Regression model
#One-hot encoding for categorical features

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss

preprocessor = ColumnTransformer(
    transformers = [
        ('numeric', StandardScaler(), numeric_features),
        ('categorical', OneHotEncoder(handle_unknown = 'ignore'), categorical_features)
    ]
)

logistic_model = Pipeline(
    steps = [
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(random_state = 99))
    ]
)

logistic_model.fit(X_train, y_train)
logistic_model_probs = logistic_model.predict_proba(X_test)[:, 1]

metrics = pd.DataFrame({
    'log_loss': [
        log_loss(y_test, baseline_probs),
        log_loss(y_test, logistic_model_probs)
    ],

    'brier_score': [
        brier_score_loss(y_test, baseline_probs),
        brier_score_loss(y_test, logistic_model_probs)
    ],

    'accuracy_at_0.5': [
        accuracy_score(y_test, baseline_probs >= 0.5),
        accuracy_score(y_test, logistic_model_probs >= 0.5)
    ]
}, index = ['Baseline Classifier', 'Logistic Model'])

metrics

,log_loss,brier_score,accuracy_at_0.5
Baseline Classifier,0.628487,0.218372,0.677870
Logistic Model,0.594849,0.201166,0.703287


In [12]:
#Spline Logistic Model with Interaction Effects

from sklearn.preprocessing import PolynomialFeatures, SplineTransformer
from sklearn.model_selection import GridSearchCV

spline_preprocessor = ColumnTransformer(
    transformers = [
        ('splines', SplineTransformer(degree = 3, knots = 'quantile'), numeric_features),
        ('onehot', OneHotEncoder(handle_unknown = 'ignore'), categorical_features)
    ]
)

spline_model = Pipeline(
    steps = [
        ('preprocessor', spline_preprocessor),
        ('model', LogisticRegression(max_iter = 1000, random_state = 99))
    ]
)

spline_param_grid = {
    'preprocessor__splines__n_knots': [3, 5, 8, 12, 16],
    'model__C': [0.01, 0.1, 1, 10, 100]
}

seasons = train_data['game_date'].dt.year.to_numpy()
train_idx = np.where(seasons < 2025)[0]
val_idx = np.where(seasons == 2025)[0]

spline_grid = GridSearchCV(
    estimator = spline_model,
    param_grid = spline_param_grid,
    scoring = 'neg_log_loss',
    n_jobs = 1,
    cv = [(train_idx, val_idx)],
)

spline_grid.fit(X_train, y_train)
spline_grid.best_params_

{'model__C': 100, 'preprocessor__splines__n_knots': 16}

In [13]:
spline_model = spline_grid.best_estimator_
spline_model_probs = spline_model.predict_proba(X_test)[:, 1]

metrics.loc['Spline Logistic Model'] = [
    log_loss(y_test, spline_model_probs),
    brier_score_loss(y_test, spline_model_probs),
    accuracy_score(y_test, spline_model_probs >= 0.5)
]

metrics

,log_loss,brier_score,accuracy_at_0.5
Baseline Classifier,0.628487,0.218372,0.677870
Logistic Model,0.594849,0.201166,0.703287
Spline Logistic Model,0.450068,0.145604,0.796867


In [14]:
#Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier

rf_numeric = ['launch_speed', 'launch_angle']
rf_features = rf_numeric + categorical_features

rf_preprocessor = ColumnTransformer(
    transformers = [
        ('numeric', 'passthrough', rf_numeric),
        ('onehot', OneHotEncoder(handle_unknown = 'ignore'), categorical_features)
    ]
)

rf_model = Pipeline(
    steps = [
        ('preprocessor', rf_preprocessor),
        ('model', RandomForestClassifier(n_estimators = 100, n_jobs = -1, random_state = 99))
    ]
)

rf_param_grid = {
    'model__min_samples_leaf': [10, 25, 50, 100, 200],
    'model__max_samples': [0.5, 0.75, None],
    'model__max_features': [0.33, 0.5, 1.0]
}

subset = train_data.sample(frac = 0.5, random_state = 99)
X_sub = subset[rf_features]
y_sub = subset['is_hit']

seasons = subset['game_date'].dt.year.to_numpy()
train_idx = np.where(seasons < 2025)[0]
val_idx = np.where(seasons == 2025)[0]

rf_grid = GridSearchCV(
    estimator = rf_model,
    param_grid = rf_param_grid,
    scoring = 'neg_log_loss',
    n_jobs = 1,
    cv = [(train_idx, val_idx)],
    refit = False
)

rf_grid.fit(X_sub, y_sub)
rf_grid.best_params_

{'model__max_features': 1.0,
 'model__max_samples': 0.5,
 'model__min_samples_leaf': 100}

In [15]:
X_train_rf = train_data[rf_features]
X_test_rf = test_data[rf_features]

rf_model.set_params(**rf_grid.best_params_)
rf_model.fit(X_train_rf, y_train)
rf_model_probs = rf_model.predict_proba(X_test_rf)[:, 1]

metrics.loc['Random Forest Classifier'] = [
    log_loss(y_test, rf_model_probs),
    brier_score_loss(y_test, rf_model_probs),
    accuracy_score(y_test, rf_model_probs >= 0.5)
]

metrics

,log_loss,brier_score,accuracy_at_0.5
Baseline Classifier,0.628487,0.218372,0.677870
Logistic Model,0.594849,0.201166,0.703287
Spline Logistic Model,0.450068,0.145604,0.796867
Random Forest Classifier,0.405898,0.131271,0.808530
